# Bronze Layer

Load the raw DT-CARGO datasets into Bronze using PySpark and Delta Lake, while preserving the source data for downstream processing.

In [7]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DT-CARGO Bronze")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    .config(
        "spark.jars.packages",
        "io.delta:delta-spark_2.13:4.0.0"
    )
    .getOrCreate()
)

print(spark.version)

4.0.1


## Fleet Data

In [8]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    DoubleType
)

In [9]:
fleet_schema = StructType([
    StructField("vehicle_id", IntegerType(), True),
    StructField("fleet_test_id", IntegerType(), True),
    StructField("gross_vehicle_weight", DoubleType(), True),
    StructField("total_mass_with_trailer", DoubleType(), True),
    StructField("axle_class", DoubleType(), True)
])

In [10]:
fleet_df = (
    spark.read
    .schema(fleet_schema)
    .option("header", True)
    .csv("/workspace/spark/local/data/raw/fleet.csv")
)

In [11]:
fleet_df.printSchema()

root
 |-- vehicle_id: integer (nullable = true)
 |-- fleet_test_id: integer (nullable = true)
 |-- gross_vehicle_weight: double (nullable = true)
 |-- total_mass_with_trailer: double (nullable = true)
 |-- axle_class: double (nullable = true)



In [12]:
print(f"Row count: {fleet_df.count()}")

fleet_df.show(5, truncate=False)

Row count: 163
+----------+-------------+--------------------+-----------------------+----------+
|vehicle_id|fleet_test_id|gross_vehicle_weight|total_mass_with_trailer|axle_class|
+----------+-------------+--------------------+-----------------------+----------+
|1         |1            |18000.0             |40000.0                |98.0      |
|2         |1            |18000.0             |40000.0                |98.0      |
|3         |1            |18000.0             |40000.0                |98.0      |
|4         |1            |18000.0             |40000.0                |98.0      |
|5         |1            |18000.0             |40000.0                |98.0      |
+----------+-------------+--------------------+-----------------------+----------+
only showing top 5 rows


In [13]:
fleet_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/workspace/spark/local/data/bronze/fleet")

26/10/11 20:44:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [14]:
fleet_bronze_df = (
    spark.read
    .format("delta")
    .load("/workspace/spark/local/data/bronze/fleet")
)

print(f"Row count: {fleet_bronze_df.count()}")

fleet_bronze_df.show(5, truncate=False)

Row count: 163


+----------+-------------+--------------------+-----------------------+----------+
|vehicle_id|fleet_test_id|gross_vehicle_weight|total_mass_with_trailer|axle_class|
+----------+-------------+--------------------+-----------------------+----------+
|1         |1            |18000.0             |40000.0                |98.0      |
|2         |1            |18000.0             |40000.0                |98.0      |
|3         |1            |18000.0             |40000.0                |98.0      |
|4         |1            |18000.0             |40000.0                |98.0      |
|5         |1            |18000.0             |40000.0                |98.0      |
+----------+-------------+--------------------+-----------------------+----------+
only showing top 5 rows


In [15]:
print(f"Number of partitions: {fleet_df.rdd.getNumPartitions()}")

Number of partitions: 1


In [16]:
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")

Default parallelism: 10


In [17]:
print(f"Fleet partitions: {fleet_df.rdd.getNumPartitions()}")

Fleet partitions: 1


## Tracks Data

In [18]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    DoubleType,
    TimestampType,
    BooleanType
)

In [19]:
tracks_schema = StructType([
    StructField("track_id", IntegerType(), True),
    StructField("vehicle_id", IntegerType(), True),
    StructField("tour_id", IntegerType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("stop_time", TimestampType(), True),
    StructField("distance", DoubleType(), True),
    StructField("track_gap", DoubleType(), True),
    StructField("avg_speed", DoubleType(), True),
    StructField("max_speed", DoubleType(), True),
    StructField("n_signal_loss", DoubleType(), True),
    StructField("d_signal_loss", DoubleType(), True),
    StructField("r_signal_loss", DoubleType(), True),
    StructField("avg_hdop", DoubleType(), True),
    StructField("home_base", BooleanType(), True),
    StructField("long_haul", BooleanType(), True),
    StructField("rest_area", BooleanType(), True),
    StructField("service_area_fuel", BooleanType(), True),
    StructField("industrial_area", BooleanType(), True),
    StructField("cid", IntegerType(), True)
])

In [20]:
tracks_df = (
    spark.read
    .schema(tracks_schema)
    .option("header", True)
    .csv("/workspace/spark/local/data/raw/tracks.csv")
)

In [21]:
tracks_df.printSchema()

root
 |-- track_id: integer (nullable = true)
 |-- vehicle_id: integer (nullable = true)
 |-- tour_id: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- stop_time: timestamp (nullable = true)
 |-- distance: double (nullable = true)
 |-- track_gap: double (nullable = true)
 |-- avg_speed: double (nullable = true)
 |-- max_speed: double (nullable = true)
 |-- n_signal_loss: double (nullable = true)
 |-- d_signal_loss: double (nullable = true)
 |-- r_signal_loss: double (nullable = true)
 |-- avg_hdop: double (nullable = true)
 |-- home_base: boolean (nullable = true)
 |-- long_haul: boolean (nullable = true)
 |-- rest_area: boolean (nullable = true)
 |-- service_area_fuel: boolean (nullable = true)
 |-- industrial_area: boolean (nullable = true)
 |-- cid: integer (nullable = true)



In [22]:
print(f"Row count: {tracks_df.count()}")

tracks_df.show(5, truncate=False)

Row count: 151598
+--------+----------+-------+-----------------------+-----------------------+--------+-----------+---------+---------+-------------+-------------+-------------+----------+---------+---------+---------+-----------------+---------------+---+
|track_id|vehicle_id|tour_id|start_time             |stop_time              |distance|track_gap  |avg_speed|max_speed|n_signal_loss|d_signal_loss|r_signal_loss|avg_hdop  |home_base|long_haul|rest_area|service_area_fuel|industrial_area|cid|
+--------+----------+-------+-----------------------+-----------------------+--------+-----------+---------+---------+-------------+-------------+-------------+----------+---------+---------+---------+-----------------+---------------+---+
|0       |1         |0      |2021-09-10 09:04:53.598|2021-09-10 10:07:57.395|30751.0 |NULL       |8.13     |20.58    |0.0          |0.0          |0.0          |0.6123537 |false    |false    |false    |false            |true           |149|
|1       |1         |0

In [23]:
print(f"Distinct track IDs: {tracks_df.select('track_id').distinct().count()}")
print(f"Distinct vehicles: {tracks_df.select('vehicle_id').distinct().count()}")
print(f"Null track IDs: {tracks_df.filter(tracks_df.track_id.isNull()).count()}")

Distinct track IDs: 151598
Distinct vehicles: 163
Null track IDs: 0


In [24]:
tracks_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/workspace/spark/local/data/bronze/tracks")

In [25]:
print(f"Tracks partitions: {tracks_df.rdd.getNumPartitions()}")

Tracks partitions: 6


In [26]:
tracks_bronze_df = (
    spark.read
    .format("delta")
    .load("/workspace/spark/local/data/bronze/tracks")
)

tracks_bronze_df.printSchema()

print(f"Row count: {tracks_bronze_df.count()}")

tracks_bronze_df.show(5, truncate=False)

root
 |-- track_id: integer (nullable = true)
 |-- vehicle_id: integer (nullable = true)
 |-- tour_id: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- stop_time: timestamp (nullable = true)
 |-- distance: double (nullable = true)
 |-- track_gap: double (nullable = true)
 |-- avg_speed: double (nullable = true)
 |-- max_speed: double (nullable = true)
 |-- n_signal_loss: double (nullable = true)
 |-- d_signal_loss: double (nullable = true)
 |-- r_signal_loss: double (nullable = true)
 |-- avg_hdop: double (nullable = true)
 |-- home_base: boolean (nullable = true)
 |-- long_haul: boolean (nullable = true)
 |-- rest_area: boolean (nullable = true)
 |-- service_area_fuel: boolean (nullable = true)
 |-- industrial_area: boolean (nullable = true)
 |-- cid: integer (nullable = true)



Row count: 151598
+--------+----------+-------+-----------------------+-----------------------+--------+-----------+---------+---------+-------------+-------------+-------------+----------+---------+---------+---------+-----------------+---------------+---+
|track_id|vehicle_id|tour_id|start_time             |stop_time              |distance|track_gap  |avg_speed|max_speed|n_signal_loss|d_signal_loss|r_signal_loss|avg_hdop  |home_base|long_haul|rest_area|service_area_fuel|industrial_area|cid|
+--------+----------+-------+-----------------------+-----------------------+--------+-----------+---------+---------+-------------+-------------+-------------+----------+---------+---------+---------+-----------------+---------------+---+
|51780   |23        |4246   |2022-01-19 12:55:10.719|2022-01-19 13:04:55.524|696.0   |10.24579578|1.19     |4.28     |54.0         |0.0          |0.0          |0.9842959 |true     |false    |false    |false            |false          |745|
|51781   |23        |4

## Speed Data

Ingest speed telemetry from the vehicle/track directory structure into the Bronze layer using Spark and Delta Lake.

In [27]:
from pyspark.sql.types import (
    StructType,
    StructField,
    DoubleType
)

speed_schema = StructType([
    StructField("epoch", DoubleType(), True),
    StructField("speed", DoubleType(), True),
    StructField("hdop", DoubleType(), True)
])

In [28]:
speed_df = (
    spark.read
    .schema(speed_schema)
    .option("header", True)
    .csv("/workspace/spark/local/data/raw/speed/1/*.csv")
)

26/10/11 20:57:59 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /workspace/spark/local/data/raw/speed/1/*.csv.
java.io.FileNotFoundException: File /workspace/spark/local/data/raw/speed/1/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at 

In [29]:
speed_df.printSchema()

root
 |-- epoch: double (nullable = true)
 |-- speed: double (nullable = true)
 |-- hdop: double (nullable = true)



In [30]:
print(f"Row count: {speed_df.count()}")

speed_df.show(5, truncate=False)

Row count: 26926736
+----------------+-----+----+
|epoch           |speed|hdop|
+----------------+-----+----+
|1.637555822847E9|0.07 |0.9 |
|1.637555822949E9|0.02 |0.9 |
|1.63755582305E9 |0.05 |0.9 |
|1.637555823152E9|0.12 |0.9 |
|1.637555823254E9|0.1  |0.9 |
+----------------+-----+----+
only showing top 5 rows


In [32]:
from pyspark.sql.functions import input_file_name, regexp_extract

In [33]:
speed_df = speed_df.withColumn(
    "source_file",
    input_file_name()
)

In [34]:
speed_df = (
    speed_df
    .withColumn(
        "vehicle_id",
        regexp_extract("source_file", r"/speed/(\d+)/", 1).cast("int")
    )
    .withColumn(
        "track_id",
        regexp_extract("source_file", r"/(\d+)\.csv$", 1).cast("int")
    )
)

In [35]:
speed_df.show(5)

+----------------+-----+----+--------------------+----------+--------+
|           epoch|speed|hdop|         source_file|vehicle_id|track_id|
+----------------+-----+----+--------------------+----------+--------+
|1.637555822847E9| 0.07| 0.9|file:///workspace...|         1|     744|
|1.637555822949E9| 0.02| 0.9|file:///workspace...|         1|     744|
| 1.63755582305E9| 0.05| 0.9|file:///workspace...|         1|     744|
|1.637555823152E9| 0.12| 0.9|file:///workspace...|         1|     744|
|1.637555823254E9|  0.1| 0.9|file:///workspace...|         1|     744|
+----------------+-----+----+--------------------+----------+--------+
only showing top 5 rows


In [36]:
speed_df.printSchema()

root
 |-- epoch: double (nullable = true)
 |-- speed: double (nullable = true)
 |-- hdop: double (nullable = true)
 |-- source_file: string (nullable = false)
 |-- vehicle_id: integer (nullable = true)
 |-- track_id: integer (nullable = true)



In [37]:
print(f"Distinct vehicles: {speed_df.select('vehicle_id').distinct().count()}")
print(f"Distinct tracks: {speed_df.select('track_id').distinct().count()}")
print(f"Null vehicle IDs: {speed_df.filter(speed_df.vehicle_id.isNull()).count()}")
print(f"Null track IDs: {speed_df.filter(speed_df.track_id.isNull()).count()}")

Distinct vehicles: 1


Distinct tracks: 2574


Null vehicle IDs: 0


Null track IDs: 0


In [38]:
print(f"Partitions: {speed_df.rdd.getNumPartitions()}")

Partitions: 85


In [39]:
print(f"Input files: {len(speed_df.inputFiles())}")

Input files: 2574


In [40]:
speed_df.explain()

== Physical Plan ==
*(1) Project [epoch#3893, speed#3894, hdop#3895, source_file#3917, cast(regexp_extract(source_file#3917, /speed/(\d+)/, 1) as int) AS vehicle_id#3918, cast(regexp_extract(source_file#3917, /(\d+)\.csv$, 1) as int) AS track_id#3919]
+- *(1) Project [epoch#3893, speed#3894, hdop#3895, input_file_name() AS source_file#3917]
   +- FileScan csv [epoch#3893,speed#3894,hdop#3895] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(2574 paths)[file:/workspace/spark/local/data/raw/speed/1/0.csv, file:/workspace..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<epoch:double,speed:double,hdop:double>


